In [ ]:
# 02_admin_ops_insights.ipynb
import pandas as pd
from datetime import datetime, timezone

from utils.io_utils import should_run, save_state, mark_state_ran, write_output_json, read_csv_to_df
from utils.feature_utils import add_date_col, get_day_name, calculate_traffic_by_slot, identify_peak_slots
from utils.metrics_utils import detect_anomalies, calculate_utilization, get_utilization_status

DATA_DIR = "data_exports"
STATE_PATH = "state/02_state.json"
OUT_PATH = "outputs/admin/ops_insights.json"

input_files = [
    f"{DATA_DIR}/visits.csv",
    f"{DATA_DIR}/queue_events.csv",
    f"{DATA_DIR}/staff_service_log.csv",
    f"{DATA_DIR}/services.csv",
    f"{DATA_DIR}/counters.csv",
]

run, new_state = should_run(input_files, STATE_PATH)
if not run:
    write_output_json(OUT_PATH, {"updated": False})
    raise SystemExit("No new CSV changes detected.")

# Load
visits = read_csv_to_df(f"{DATA_DIR}/visits.csv", date_cols=["timestamp"])
events = read_csv_to_df(f"{DATA_DIR}/queue_events.csv", date_cols=["event_time"])
staff_log = read_csv_to_df(f"{DATA_DIR}/staff_service_log.csv", date_cols=["start_time","end_time"])
services = read_csv_to_df(f"{DATA_DIR}/services.csv")
counters = read_csv_to_df(f"{DATA_DIR}/counters.csv")

visits = add_date_col(visits, "timestamp", "date")
period = {
    "start": str(pd.to_datetime(visits["date"]).min().date()),
    "end": str(pd.to_datetime(visits["date"]).max().date())
}

# -------------------------
# Peak hours (heatmap + top slots)
# -------------------------
traffic_df = calculate_traffic_by_slot(visits)
heatmap = traffic_df.copy()
heatmap["dow_name"] = heatmap["dow"].apply(lambda d: get_day_name(int(d)))
heatmap_records = heatmap.sort_values(["dow","hour"])[["dow","dow_name","hour","avg_traffic"]].to_dict("records")
top_slots_records = identify_peak_slots(traffic_df, top_n=10)

# -------------------------
# Drop-off periods (by hour + by service)
# Drop-off = cancelled + no_show (per your contract)
# -------------------------
visits["is_dropoff"] = visits["status"].isin(["cancelled","no_show"]).astype(int)

by_hour = visits.groupby("hour", as_index=False).agg(
    total=("visit_id","count"),
    dropoff_count=("is_dropoff","sum")
)
by_hour["dropoff_rate"] = (by_hour["dropoff_count"] / by_hour["total"]).fillna(0).round(3)
dropoff_by_hour = by_hour.sort_values("hour")[["hour","dropoff_count","dropoff_rate"]].to_dict("records")

by_service = visits.groupby(["service_id","service_name"], as_index=False).agg(
    total=("visit_id","count"),
    dropoff_count=("is_dropoff","sum")
)
by_service["dropoff_rate"] = (by_service["dropoff_count"] / by_service["total"]).fillna(0).round(3)
dropoff_by_service = (by_service.sort_values("dropoff_rate", ascending=False)
                      .head(20)[["service_id","service_name","dropoff_rate"]].to_dict("records"))

# -------------------------
# Best times to visit (slot score = low traffic + short wait)
# -------------------------
slot = visits.groupby(["dow","hour"], as_index=False).agg(
    total_visits=("visit_id","count"),
    avg_wait=("wait_time_minutes","mean")
)
slot["traffic_rank"] = slot["total_visits"].rank(pct=True)
slot["wait_rank"] = slot["avg_wait"].fillna(0).rank(pct=True)
slot["score"] = ((1 - slot["traffic_rank"]) * 0.6 + (1 - slot["wait_rank"]) * 0.4) * 100
slot["score"] = slot["score"].round(0).astype(int)
slot["dow_name"] = slot["dow"].apply(lambda d: get_day_name(int(d)))

best_slots = slot.sort_values("score", ascending=False).head(12).copy()
best_slots["reason"] = "low traffic + short wait"
best_times_records = best_slots[["dow","dow_name","hour","score","reason"]].to_dict("records")

# -------------------------
# Weekly anomalies (daily total arrivals)
# -------------------------
daily = visits.groupby("date", as_index=False).agg(total_arrivals=("visit_id","count"))
anomalies = detect_anomalies(daily, value_col="total_arrivals", date_col="date", threshold=2.0)

# -------------------------
# Service usage ranking
# -------------------------
total = max(len(visits), 1)
usage = visits.groupby(["service_id","service_name"], as_index=False).agg(total_visits=("visit_id","count"))
usage["percentage"] = (usage["total_visits"] / total * 100).round(1)
usage = usage.sort_values("total_visits", ascending=False).reset_index(drop=True)
usage["rank"] = usage.index + 1
service_usage_records = usage.head(50)[["service_id","service_name","total_visits","percentage","rank"]].to_dict("records")

# -------------------------
# Resource efficiency (utilization per service)
# -------------------------
busy = staff_log.groupby("service_id", as_index=False).agg(busy_minutes=("duration_minutes","sum"))

active_counters = counters[counters["is_active"] == True].groupby("service_id", as_index=False).agg(
    active_counters=("counter_id","count")
)

eff = busy.merge(active_counters, on="service_id", how="left")
eff["active_counters"] = eff["active_counters"].fillna(1)

start = visits["timestamp"].min()
end = visits["timestamp"].max()
period_minutes = max((end - start).total_seconds() / 60.0, 1)

eff["utilization"] = eff.apply(
    lambda r: calculate_utilization(r["busy_minutes"], period_minutes * r["active_counters"]),
    axis=1
)

svc_map = services[["service_id","service_name"]]
eff = eff.merge(svc_map, on="service_id", how="left")
eff["status"] = eff["utilization"].apply(get_utilization_status)
resource_eff_records = eff[["service_id","service_name","utilization","status"]].to_dict("records")

# -------------------------
# Recommendations (simple rule-based MVP)
# -------------------------
recs = []

worst_drop = by_hour.sort_values("dropoff_rate", ascending=False).head(1)
if len(worst_drop):
    h = int(worst_drop.iloc[0]["hour"])
    rate = float(worst_drop.iloc[0]["dropoff_rate"])
    recs.append(f"Drop-offs peak around {h}:00 (rate {rate}). Consider SMS reminders or clearer queue expectations during that hour.")

over = eff[eff["status"] == "under_resourced"].sort_values("utilization", ascending=False).head(2)
for _, r in over.iterrows():
    recs.append(f"{r['service_name']} is under-resourced (utilization {round(float(r['utilization']),2)}). Consider adding capacity during peak periods.")

if best_times_records:
    bt = best_times_records[0]
    recs.append(f"Encourage visits on {bt['dow_name']} at {bt['hour']}:00 for smoother traffic.")

payload = {
  "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00","Z"),
  "period": period,
  "peak_hours": {"heatmap": heatmap_records, "top_slots": top_slots_records},
  "dropoff_periods": {"by_hour": dropoff_by_hour, "by_service": dropoff_by_service},
  "best_times": {"recommended_slots": best_times_records},
  "anomalies": anomalies,
  "service_usage": {"ranked": service_usage_records},
  "resource_efficiency": resource_eff_records,
  "recommendations": recs
}

write_output_json(OUT_PATH, payload)
save_state(STATE_PATH, mark_state_ran(new_state))

print("✅ 02 complete: outputs/admin/ops_insights.json written")
